# 09 - Cross-validation of the main models

**Validation design.** The 56 training clips are split into 3 folds with `GroupKFold` by clip
(notebook 04), so all frames of a clip are in the same fold and no clip is seen in training and
evaluation at once. Run k of an experiment trains on the other two folds (every 10th frame,
about 1630 frames) and is scored on the holdout of fold k (every 10th frame, about 815 frames,
original frames without masking). The official `val` split (7 clips) is scored by every run as
an extra, fully independent check. There is no learned preprocessing: masking of ignored regions
uses the annotations of the frame itself, and normalisation is fixed by the pretrained models.

**Out-of-fold score.** The holdout predictions of the 3 runs are pooled and scored once against
all 2446 holdout frames, so every training clip is scored exactly once by a model that never saw
it. The mean and standard deviation over the folds show how much a result depends on the fold.

**What is cross-validated.** The initial and the tuned version of both models of this part
(YOLO11s: af0 at 640 px, af1 at 1152 px; RT-DETR-l: qb0 with defaults, qb3 with the paper
recipe), and the inference steps of the final pipeline on qb3. The single-change ablations of
notebooks 05-08 (NWD, top-k, queries, P2, ...) are run on fold 0 only: each RT-DETR run takes
about 3 hours on our GPU, and their effect is tested with the per-clip paired test on 26 clips.
This notebook checks that the main conclusions do not depend on fold 0.

Commands (from the repository root):

```
python scripts/prepare_data.py --fold 1 --train-step 10
python scripts/prepare_data.py --fold 2 --train-step 10
python scripts/train.py configs/query_based/qb3_bblr_nobiaswarm_1152.yaml --fold 1
python scripts/temporal.py configs/query_based/qb3_bblr_nobiaswarm_1152.yaml --strategy tiles --fold 1
```

(the same for fold 2 and for af0, af1, qb0; fold 0 is the original run of notebooks 05 and 06)

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.analysis import paired_comparison, per_clip_ap
from src.common import N_FOLDS, REPO_ROOT, RUNS_DIR
from src.cross_validation import cross_validate, oof_ground_truth
from src.inference import from_coco, to_coco
from src.predict import ALL_SETS
from src.temporal import frames_table, neighbours, rescore
from src.train_tools import fold_config, load_config, run_dir

MODELS = {
    "af0 YOLO11s 640 (initial)": "configs/anchor_free/af0_baseline_640.yaml",
    "af1 YOLO11s 1152 (tuned)": "configs/anchor_free/af1_imgsz1152.yaml",
    "qb0 RT-DETR-l defaults (initial)": "configs/query_based/qb0_rtdetr_l_1152.yaml",
    "qb3 RT-DETR-l paper recipe (tuned)": "configs/query_based/qb3_bblr_nobiaswarm_1152.yaml",
}
pd.set_option("display.precision", 4)

## 1. Per-fold and out-of-fold results

In [2]:
def fold_runs(config_path):
    # Finished runs of one experiment: fold -> run directory
    cfg = load_config(REPO_ROOT / config_path)
    found = {}
    for k in range(N_FOLDS):
        directory = run_dir(fold_config(cfg, k))
        if (directory / "metrics.json").exists():
            found[k] = directory
    return found


def holdout_predictions(runs):
    return {k: json.loads((directory / f"predictions_fold{k}.json").read_text()) for k, directory in runs.items()}


rows, predictions = [], {}
for model, config_path in MODELS.items():
    runs = fold_runs(config_path)
    if not runs:
        continue
    predictions[model] = holdout_predictions(runs)
    cv = cross_validate(predictions[model])
    val_ap = [json.loads((d / "metrics.json").read_text())["val"]["AP"] for d in runs.values()]
    row = {"model": model, "folds": len(runs)}
    row.update({f"AP_fold{k}": metrics["AP"] for k, metrics in cv["folds"].items()})
    row.update({"AP_mean": cv["mean"]["AP"], "AP_std": cv["std"]["AP"]})
    if cv["oof"]:
        row.update({f"oof_{m}": cv["oof"][m] for m in ["AP", "AP50", "AP_small", "AP_medium", "AP_large"]})
    row.update({"val_AP_mean": pd.Series(val_ap).mean(), "val_AP_std": pd.Series(val_ap).std()})
    rows.append(row)
cv_table = pd.DataFrame(rows).set_index("model")
cv_table

,folds,AP_fold0,AP_fold1,AP_fold2,AP_mean,AP_std,oof_AP,oof_AP50,oof_AP_small,oof_AP_medium,oof_AP_large,val_AP_mean,val_AP_std
model,,,,,,,,,,,,,
af0 YOLO11s 640 (initial),3,0.1683,0.1303,0.1241,0.1409,0.0239,0.1373,0.2640,0.0310,0.1212,0.2845,0.1266,0.0144
af1 YOLO11s 1152 (tuned),3,0.2292,0.1818,0.1720,0.1943,0.0306,0.1897,0.3555,0.0658,0.1811,0.3344,0.1813,0.0201
qb0 RT-DETR-l defaults (initial),3,0.2956,0.2411,0.2320,0.2562,0.0344,0.2457,0.4612,0.1063,0.2284,0.4066,0.2420,0.0190
qb3 RT-DETR-l paper recipe (tuned),3,0.2988,0.2594,0.2467,0.2683,0.0271,0.2602,0.4864,0.1338,0.2437,0.4318,0.2626,0.0154


## 2. Do the conclusions hold in every fold?

For each comparison of notebooks 05 and 06: the AP difference in every fold, and the paired
per-clip test on all 56 training clips, each scored out of fold (19 + 19 + 18 clips; the paired
test of the earlier notebooks had 26 clips).

In [3]:
COMPARISONS = [
    ("af1 YOLO11s 1152 (tuned)", "af0 YOLO11s 640 (initial)"),
    ("qb3 RT-DETR-l paper recipe (tuned)", "qb0 RT-DETR-l defaults (initial)"),
    ("qb0 RT-DETR-l defaults (initial)", "af1 YOLO11s 1152 (tuned)"),
    ("qb3 RT-DETR-l paper recipe (tuned)", "af1 YOLO11s 1152 (tuned)"),
]

oof_gt = oof_ground_truth()
oof_clip_ap = {model: per_clip_ap(oof_gt, [d for dets in preds.values() for d in dets])
               for model, preds in predictions.items() if len(preds) == N_FOLDS}

rows = []
for new, old in COMPARISONS:
    if new not in cv_table.index or old not in cv_table.index:
        continue
    row = {"comparison": f"{new.split()[0]} vs {old.split()[0]}"}
    for k in range(N_FOLDS):
        if f"AP_fold{k}" in cv_table:
            row[f"diff_fold{k}"] = cv_table.loc[new, f"AP_fold{k}"] - cv_table.loc[old, f"AP_fold{k}"]
    if new in oof_clip_ap and old in oof_clip_ap:
        paired = paired_comparison({old: oof_clip_ap[old], new: oof_clip_ap[new]}, reference=old).loc[new]
        row.update({"paired_diff": paired["mean_diff"], "std_err": paired["std_err"],
                    "clips_better": int(paired["clips_better"]), "clips_worse": int(paired["clips_worse"]),
                    "significant": abs(paired["mean_diff"]) >= 2 * paired["std_err"]})
    rows.append(row)
pd.DataFrame(rows).set_index("comparison")

,diff_fold0,diff_fold1,diff_fold2,paired_diff,std_err,clips_better,clips_worse,significant
comparison,,,,,,,,
af1 vs af0,0.0609,0.0515,0.0479,0.0565,0.0055,52,4,True
qb3 vs qb0,0.0031,0.0183,0.0147,0.0119,0.0046,36,20,True
qb0 vs af1,0.0664,0.0593,0.0599,0.0617,0.0048,52,4,True
qb3 vs af1,0.0695,0.0776,0.0747,0.0736,0.0057,54,2,True


## 3. The final pipeline out of fold

The inference steps of notebook 07 on the qb3 runs of every fold: the plain pass, full frame +
2x2 tiles, and tiles + temporal rescoring (`smooth`, radius 2, IoU 0.3, the setting chosen in
notebook 07 on fold 0 and val). Folds 1 and 2 were not used for any choice, so they are a clean
test of these choices.

In [4]:
QB3 = "configs/query_based/qb3_bblr_nobiaswarm_1152.yaml"
TEMPORAL_SETTING = {"radius": 2, "iou_thr": 0.3, "mode": "smooth"}


def temporal_dir(fold):
    cfg = fold_config(load_config(REPO_ROOT / QB3), fold)
    return RUNS_DIR / ("temporal" if fold == 0 else "temporal_cv") / f"{cfg['name']}_tiles_{cfg['imgsz']}"


steps = {"plain": {}, "tiles": {}, "tiles + temporal": {}}
qb3_runs = fold_runs(QB3)
for k in range(N_FOLDS):
    path = temporal_dir(k) / f"detections_fold{k}.json"
    if k not in qb3_runs or not path.exists():
        continue
    split, gt_path = ALL_SETS[f"fold{k}"]
    centre_ids = [img["id"] for img in json.loads(gt_path.read_text())["images"]]
    preds = from_coco(json.loads(path.read_text()))
    window = neighbours(frames_table(split), centre_ids, TEMPORAL_SETTING["radius"])
    rescored = rescore(preds, window, **TEMPORAL_SETTING)

    steps["plain"][k] = json.loads((qb3_runs[k] / f"predictions_fold{k}.json").read_text())
    steps["tiles"][k] = [det for i in centre_ids if i in preds for det in to_coco(preds[i], i)]
    steps["tiles + temporal"][k] = [det for i, pred in rescored.items() for det in to_coco(pred, i)]

pipeline_rows, pipeline_clip_ap = [], {}
for step, by_fold in steps.items():
    if not by_fold:
        continue
    cv = cross_validate(by_fold)
    row = {"step": step, **{f"AP_fold{k}": m["AP"] for k, m in cv["folds"].items()},
           "AP_mean": cv["mean"]["AP"], "AP_std": cv["std"]["AP"]}
    if cv["oof"]:
        row.update({"oof_AP": cv["oof"]["AP"], "oof_AP50": cv["oof"]["AP50"], "oof_AP_small": cv["oof"]["AP_small"]})
        pipeline_clip_ap[step] = per_clip_ap(oof_gt, [d for dets in by_fold.values() for d in dets])
    pipeline_rows.append(row)
if pipeline_rows:
    display(pd.DataFrame(pipeline_rows).set_index("step"))

if len(pipeline_clip_ap) == 3:
    for new, old in [("tiles", "plain"), ("tiles + temporal", "tiles")]:
        comparison = paired_comparison({old: pipeline_clip_ap[old], new: pipeline_clip_ap[new]}, reference=old)
        comparison["significant"] = comparison["mean_diff"].abs() >= 2 * comparison["std_err"]
        print(f"{new} against {old} (56 clips, out of fold)")
        display(comparison.drop(index=old))

,AP_fold0,AP_fold1,AP_fold2,AP_mean,AP_std,oof_AP,oof_AP50,oof_AP_small
step,,,,,,,,
plain,0.2988,0.2594,0.2467,0.2683,0.0271,0.2602,0.4864,0.1338
tiles,0.3246,0.2789,0.2680,0.2905,0.0300,0.2800,0.5258,0.1450
tiles + temporal,0.3315,0.2824,0.2740,0.2960,0.0311,0.2860,0.5363,0.1491


tiles against plain (56 clips, out of fold)


,mean_diff,std_err,clips_better,clips_worse,significant
tiles,0.0158,0.0042,48.0,8.0,True


tiles + temporal against tiles (56 clips, out of fold)


,mean_diff,std_err,clips_better,clips_worse,significant
tiles + temporal,0.0051,0.0009,45.0,11.0,True


## 4. Tiles and the frame width

Notebook 07 keeps the tiles only on frames narrower than 1600 px, a rule that comes from the public
leaderboard. All frames of a clip have the same width, so the rule can be tested here: the gain of
the tiles per clip, grouped by width, and the score of tiles on narrow frames only or on wide
frames only.

In [5]:
from PIL import Image

NARROW_WIDTH = 1600

train_frames = frames_table("train")
clip_of = dict(zip(train_frames["image_id"], train_frames["clip"]))
clip_width = {clip: Image.open(group["path"].iloc[0]).size[0] for clip, group in train_frames.groupby("clip")}


def tiles_on(narrow):
    # Tiles on the frames of one width group, the plain pass on the others
    mixed = {}
    for k in steps["plain"]:
        is_tiled = lambda det: (clip_width[clip_of[det["image_id"]]] < NARROW_WIDTH) == narrow
        mixed[k] = ([det for det in steps["plain"][k] if not is_tiled(det)]
                    + [det for det in steps["tiles"][k] if is_tiled(det)])
    return mixed


width_steps = {"plain": steps["plain"], "tiles everywhere": steps["tiles"],
               "tiles on narrow frames": tiles_on(True), "tiles on wide frames": tiles_on(False)}
rows, clip_ap = [], {}
for name, dets in width_steps.items():
    cv = cross_validate(dets)
    rows.append({"step": name, "AP_mean": cv["mean"]["AP"], "AP_oof": cv["oof"]["AP"]})
    clip_ap[name] = per_clip_ap(oof_gt, [d for fold in dets.values() for d in fold])
display(pd.DataFrame(rows).set_index("step"))

for new in ["tiles on narrow frames", "tiles on wide frames"]:
    paired = paired_comparison({"tiles everywhere": clip_ap["tiles everywhere"], new: clip_ap[new]},
                               reference="tiles everywhere")
    display(paired.loc[[new]])

gain = pd.DataFrame(clip_ap)
gain = pd.DataFrame({"gain": gain["tiles everywhere"] - gain["plain"],
                     "width": [clip_width[clip] for clip in gain.index]})
gain["group"] = gain["width"].lt(NARROW_WIDTH).map({True: "narrow", False: "wide"})
display(gain.groupby("width")["gain"].agg(["mean", "sem", "size"]))
display(gain.groupby("group")["gain"].agg(["mean", "sem", "size"]))

,AP_mean,AP_oof
step,,
plain,0.2683,0.2602
tiles everywhere,0.2905,0.2800
tiles on narrow frames,0.2705,0.2622
tiles on wide frames,0.2871,0.2767


,mean_diff,std_err,clips_better,clips_worse
tiles on narrow frames,-0.0143,0.0042,5.0,39.0


,mean_diff,std_err,clips_better,clips_worse
tiles on wide frames,-0.0015,0.0011,3.0,9.0


,mean,sem,size
width,,,
1344,0.0065,0.0054,11
1360,0.0137,NaN,1
1904,0.0187,0.0045,30
1920,0.0288,NaN,1
2688,0.0154,0.0170,11
2720,0.0207,0.0097,2


,mean,sem,size
group,,,
narrow,0.0071,0.0050,12
wide,0.0182,0.0051,44


## 5. Findings

**Models.** Fold 0 is the easiest fold for every model, fold 2 the hardest, so the spread between
the folds comes mostly from the clips. The order of the models is the same in all folds.

| Comparison, 56 clips out of fold | Paired difference | Clips better / worse |
|---|---|---|
| af1 vs af0 (input 640 to 1152 px) | +0.0565 +- 0.0055 | 52 / 4 |
| qb0 vs af1 (YOLO11s to RT-DETR-l) | +0.0617 +- 0.0048 | 52 / 4 |
| qb3 vs qb0 (paper recipe, no bias warm-up) | +0.0119 +- 0.0046 | 36 / 20 |

The three main conclusions of notebooks 05 and 06 came from fold 0 and val. They hold on all 56
clips, and 37 of them were not used for any choice. The out-of-fold AP uses the saved predictions
(score at least 0.05), so it is a little lower than a full evaluation, most of all for YOLO11s.
The differences above are much larger than this effect.

**Pipeline.** Tiles add +0.0158 +- 0.0042 (48 / 8 clips) and temporal rescoring adds
+0.0051 +- 0.0009 (45 / 11) on top of them. Both were chosen on fold 0 and val, and both hold on
folds 1 and 2. The out-of-fold AP of qb3 goes from 0.2602 to 0.2800 and then to 0.2860.

**Frame width.** Tiles help wide clips (+0.0182 +- 0.0051, 39 of 44 clips better) more than narrow
ones (+0.0071 +- 0.0050, 9 of 12). Tiles on narrow frames only are significantly worse than tiles
everywhere (-0.0143 +- 0.0042, 5 / 39). So the rule of notebook 07 has no support in our folds, and
the public leaderboard shows the opposite effect. Our folds cover this case poorly: they have 12
narrow clips and no frames of 680 or 960 px, while 33 of the 48 test clips are narrow. This is why
the second final submission uses tiles on all frames.

**Run differences.** af1 folds 1 and 2 use batch 4 instead of 8 because of an out-of-memory error
on dense frames, and fold 1 changed from batch 8 to 4 at epoch 8. The optimisation step is the same
because gradients are accumulated to the nominal batch 64. The last runs use one data loader worker
to save memory, which changes only the speed of data loading.